# Multimodal Deepfake Detection for Kaggle: Video Swin Tiny + WavLM-Base+ + Cross-Attention Fusion

This notebook is the Kaggle-compatible version of the new research approach. It starts with a 970-clip pilot and keeps a clean path toward full-dataset training.

Training plan:
- Stage A: frozen unimodal training
- Stage B: partial unfreezing
- Stage C: fusion training with partially frozen encoders
- Stage D: end-to-end low-learning-rate fine-tuning

In [ ]:
from __future__ import annotations

import json
import os
import random
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import FileLink, display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.amp.autocast_mode import autocast
from torch.amp.grad_scaler import GradScaler
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import subprocess
import tempfile

try:
    import timm
except ImportError:
    timm = None

try:
    from transformers import WavLMModel
except ImportError:
    WavLMModel = None

try:
    import torchaudio
except Exception:
    torchaudio = None

try:
    import scipy.io.wavfile as scipy_wavfile
except Exception:
    scipy_wavfile = None

print('Imports ready')

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_project_root() -> Path:
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return Path('/kaggle/working')
    return Path(r'F:\Sixth Semester\DEEPfake Papers\FakeBDTeen')


def detect_dataset_root() -> Path:
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates = [p for p in kaggle_input.iterdir() if p.is_dir()]
        for candidate in candidates:
            if list(candidate.rglob('*.mp4')):
                return candidate
    return resolve_project_root() / 'FakeBDTeen'


PROJECT_ROOT = resolve_project_root()
RAW_DATASET_ROOT = detect_dataset_root()
WORK_DIR = PROJECT_ROOT / 'outputs_video_swin_wavlm_kaggle'
WORK_DIR.mkdir(parents=True, exist_ok=True)

PILOT_CLIPS = 960
FULL_DATASET_MODE = False
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONFIG = {
    'mode': 'pilot' if not FULL_DATASET_MODE else 'full',
    'pilot_clips': PILOT_CLIPS,
    'full_dataset_enabled': FULL_DATASET_MODE,
    'seed': SEED,
    'num_frames': 16,
    'frame_size': 224,
    'audio_sample_rate': 16000,
    'batch_size': 4,
    'num_workers': 2,
    'accumulation_steps': 4,
    'epochs_stage_a': 4,
    'epochs_stage_b': 3,
    'epochs_stage_c': 5,
    'epochs_stage_d': 3,
    'lr_stage_a': 1e-4,
    'lr_stage_b': 8e-5,
    'lr_stage_c': 5e-5,
    'lr_stage_d': 1e-5,
    'weight_decay': 1e-4,
    'use_amp': torch.cuda.is_available(),
    'resume_from_checkpoints': True,
    'checkpoint_every_epoch': True,
    'class_names': ['FF', 'FR', 'RF', 'RR'],
    'video_model_name': 'swin_tiny_patch4_window7_224',
    'audio_model_name': 'microsoft/wavlm-base-plus',
}

set_seed(SEED)
print('Project root:', PROJECT_ROOT)
print('Dataset root:', RAW_DATASET_ROOT)
print('Working directory:', WORK_DIR)
print('Device:', DEVICE)

## 1. Import Required Libraries and Dependencies

This notebook uses PyTorch, timm, transformers, OpenCV, NumPy, pandas, and scikit-learn.

## 2. Set Up Configuration and Hyperparameters

The notebook keeps a pilot/full switch so the same structure can scale later.

In [ ]:
def normalize_path_text(text: str) -> str:
    # normalize underscores and spacing, then map known labels to a canonical form
    s = text.replace('_', ' ')
    # ensure spaces around plus signs
    s = s.replace('+', ' + ')
    s = ' '.join(s.split())
    lower = s.lower()
    # targeted canonical replacements
    if 'fake video' in lower:
        s = s.replace(s[s.lower().find('fake video'):s.lower().find('fake video') + len('fake video')], 'Fake Video')
    if 'real video' in lower:
        s = s.replace(s[s.lower().find('real video'):s.lower().find('real video') + len('real video')], 'Real Video')
    if 'fake audio' in lower:
        s = s.replace(s[s.lower().find('fake audio'):s.lower().find('fake audio') + len('fake audio')], 'Fake Audio')
    if 'real audio' in lower:
        s = s.replace(s[s.lower().find('real audio'):s.lower().find('real audio') + len('real audio')], 'Real Audio')
    # collapse extra spaces again
    return ' '.join(s.split())


def infer_class_name(path_text: str) -> str:
    normalized = normalize_path_text(path_text)
    if 'Fake Video + Fake Audio' in normalized:
        return 'FF'
    if 'Fake Video + Real Audio' in normalized:
        return 'FR'
    if 'Real Video + Fake Audio' in normalized:
        return 'RF'
    if 'Real Video + Real Audio' in normalized:
        return 'RR'
    raise ValueError(f'Unable to infer class from path: {path_text}')


def discover_clip_records(root: Path) -> List[Dict[str, str]]:
    records: List[Dict[str, str]] = []
    for mp4_path in root.rglob('*.mp4'):
        relative_path = mp4_path.relative_to(root)
        normalized_relative = Path(*[normalize_path_text(part) for part in relative_path.parts])
        records.append({
            'path': str(mp4_path),
            'relative_path': str(normalized_relative),
            'speaker_id': normalized_relative.parts[-2] if len(normalized_relative.parts) >= 2 else 'unknown',
            'label': infer_class_name(str(normalized_relative)),
        })
    return records


all_records = discover_clip_records(RAW_DATASET_ROOT)
print('Total clips discovered:', len(all_records))
print('Unique speakers:', len({record['speaker_id'] for record in all_records}))

## 3. Load and Preprocess Dataset (970 Clips)

The pilot subset is created from speaker groups so the same speaker does not leak across splits.

In [ ]:
def build_pilot_per_folder(records: Sequence[Dict[str, str]], per_folder: int = 4, seed: int = 42) -> List[Dict[str, str]]:
    """Select `per_folder` clips from each top-level folder group (grouping by first two path parts).
    If a group has fewer than `per_folder` clips, take all available clips from that group.
    Saves a manifest named `pilot_manifest_per_folder_{per_folder}.json` in WORK_DIR.
    """
    rng = random.Random(seed)
    groups: Dict[str, List[Dict[str, str]]] = defaultdict(list)
    for record in records:
        rel = Path(record['relative_path'])
        if len(rel.parts) >= 2:
            key = f"{rel.parts[0]}/{rel.parts[1]}"
        elif len(rel.parts) >= 1:
            key = rel.parts[0]
        else:
            key = 'ungrouped'
        groups[key].append(record)

    selected: List[Dict[str, str]] = []
    for k, items in groups.items():
        rng.shuffle(items)
        take = min(per_folder, len(items))
        selected.extend(items[:take])

    manifest_path = WORK_DIR / f'pilot_manifest_per_folder_{per_folder}.json'
    with manifest_path.open('w', encoding='utf-8') as f:
        json.dump(selected, f, indent=2)
    print('Saved per-folder pilot manifest to', manifest_path)
    print('Pilot clips selected (total):', len(selected), 'groups:', len(groups))
    print('Pilot class distribution:', Counter(item['label'] for item in selected))
    return selected


# Build pilot by taking 4 clips per top-level folder
pilot_records = build_pilot_per_folder(all_records, per_folder=4, seed=SEED)
print('Pilot speakers:', len({item['speaker_id'] for item in pilot_records}))

## 4. Implement Video Preprocessing Pipeline

This section defines frame extraction, resizing, normalization, and augmentation hooks.

In [ ]:
def extract_video_frames(video_path: str, num_frames: int = 8, size: int = 224) -> torch.Tensor:
    capture = cv2.VideoCapture(video_path)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        capture.release()
        return torch.zeros(num_frames, 3, size, size)
    frame_indices = np.linspace(0, frame_count - 1, num_frames).astype(int)
    frames = []
    for frame_index in frame_indices:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = capture.read()
        if not ok:
            frame = np.zeros((size, size, 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (size, size), interpolation=cv2.INTER_AREA)
        frame = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
        frames.append(frame)
    capture.release()
    return torch.stack(frames, dim=0)


def preprocess_video_clip(video_path: str, train_mode: bool = True) -> torch.Tensor:
    frames = extract_video_frames(video_path, num_frames=CONFIG['num_frames'], size=CONFIG['frame_size'])
    if train_mode:
        frames = apply_video_augmentations(frames)
    return frames


def apply_video_augmentations(frames: torch.Tensor) -> torch.Tensor:
    if frames.size(0) > 1 and random.random() < 0.5:
        shift = random.randint(0, frames.size(0) - 1)
        frames = torch.roll(frames, shifts=shift, dims=0)
    if random.random() < 0.5:
        frames = frames.flip(dims=[3])
    return frames


print('Video preprocessing ready')

## 5. Implement Audio Preprocessing Pipeline

This section defines waveform loading and lightweight audio augmentation hooks for WavLM.

In [ ]:
def load_audio_waveform(video_path: str, sample_rate: int = 16000, duration: float = 10.0) -> torch.Tensor:
    """Extract audio from a video using ffmpeg then load with torchaudio or scipy fallback.
    Returns a 1 x N torch.FloatTensor sampled at `sample_rate` and trimmed/padded to `duration` seconds.
    """
    tmp_file = Path(tempfile.NamedTemporaryFile(suffix='.wav', delete=False).name)
    cmd = [
        'ffmpeg', '-y', '-i', str(video_path),
        '-ar', str(sample_rate), '-ac', '1', '-t', str(duration), str(tmp_file)
    ]
    try:
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except Exception as e:
        print('ffmpeg extraction failed, returning zeros:', e)
        return torch.zeros(1, int(sample_rate * duration))

    waveform = None
    if torchaudio is not None:
        try:
            wav, sr = torchaudio.load(str(tmp_file))
            if sr != sample_rate:
                wav = torchaudio.transforms.Resample(sr, sample_rate)(wav)
            waveform = wav
        except Exception:
            waveform = None
    if waveform is None and scipy_wavfile is not None:
        try:
            sr, data = scipy_wavfile.read(str(tmp_file))
            data = np.asarray(data, dtype=np.float32)
            if data.ndim > 1:
                data = data.mean(axis=1)
            # normalize if integers
            if data.dtype != np.float32:
                maxv = max(1.0, np.max(np.abs(data)))
                data = data.astype(np.float32) / maxv
            waveform = torch.from_numpy(data).unsqueeze(0)
            if sr != sample_rate:
                # simple resample via linear interpolation
                new_len = int(round(waveform.size(1) * (sample_rate / sr)))
                interp = np.interp(np.linspace(0, waveform.size(1) - 1, new_len), np.arange(waveform.size(1)), waveform.numpy().squeeze())
                waveform = torch.from_numpy(interp.astype(np.float32)).unsqueeze(0)
        except Exception:
            waveform = None

    if waveform is None:
        waveform = torch.zeros(1, int(sample_rate * duration))

    expected = int(sample_rate * duration)
    if waveform.size(1) < expected:
        waveform = F.pad(waveform, (0, expected - waveform.size(1)))
    elif waveform.size(1) > expected:
        waveform = waveform[:, :expected]

    try:
        tmp_file.unlink()
    except Exception:
        pass
    return waveform


def preprocess_audio_clip(video_path: str, train_mode: bool = True) -> torch.Tensor:
    waveform = load_audio_waveform(video_path, sample_rate=CONFIG['audio_sample_rate'])
    if train_mode:
        waveform = apply_audio_augmentations(waveform)
    return waveform


def apply_audio_augmentations(waveform: torch.Tensor) -> torch.Tensor:
    # small additive noise and random time-shift
    if random.random() < 0.5:
        waveform = waveform + 0.001 * torch.randn_like(waveform)
    if random.random() < 0.3:
        shift = int(0.1 * CONFIG['audio_sample_rate'])
        roll = random.randint(-shift, shift)
        waveform = torch.roll(waveform, shifts=roll, dims=1)
    return waveform


print('Audio preprocessing ready (torchaudio={} scipy={})'.format(torchaudio is not None, scipy_wavfile is not None))

## 6. Build Video Swin Tiny Backbone

The notebook prefers a Swin Tiny family model from timm.

In [ ]:
def resolve_video_swin_model_name() -> str:
    if timm is None:
        raise ImportError('timm is required for the video backbone.')
    video_models = timm.list_models('*video*swin*tiny*')
    if video_models:
        return video_models[0]
    swin_models = timm.list_models('*swin*tiny*')
    if swin_models:
        return swin_models[0]
    return 'swin_tiny_patch4_window7_224'


class VideoSwinTinyBackbone(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        if timm is None:
            raise ImportError('timm is required for the video backbone.')
        self.model_name = resolve_video_swin_model_name()
        self.model = timm.create_model(self.model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.feature_dim = getattr(self.model, 'num_features', 768)

        # Temporal fusion layers should be initialized here (not in forward)
        # so they are tracked by optimizer and saved in checkpoints.
        self.temporal_dim = max(64, self.feature_dim // 2)
        self.temporal_proj = nn.Linear(self.feature_dim, self.temporal_dim)
        self.temporal_attn = nn.MultiheadAttention(self.temporal_dim, num_heads=4, batch_first=True)
        self.temporal_ln = nn.LayerNorm(self.temporal_dim)

    def forward(self, frames: torch.Tensor) -> torch.Tensor:
        # Expect frames: [B, T, C, H, W]
        batch_size, num_frames, channels, height, width = frames.shape

        # Per-frame Swin encoding
        frames = frames.view(batch_size * num_frames, channels, height, width)
        per_frame = self.model(frames)  # [B*T, F]
        per_frame = per_frame.view(batch_size, num_frames, -1)  # [B, T, F]

        # Temporal attention pooling across frame tokens
        proj = self.temporal_proj(per_frame)  # [B, T, D]
        attn_out, _ = self.temporal_attn(proj, proj, proj)
        pooled = attn_out.mean(dim=1)
        pooled = self.temporal_ln(pooled)
        return pooled


print('Video Swin Tiny backbone ready')

## 7. Build WavLM-Base+ Audio Backbone

The audio backbone uses WavLM-Base+ for robust speech embeddings.

In [ ]:
class WavLMBasePlusBackbone(nn.Module):
    def __init__(self, model_name: str = 'microsoft/wavlm-base-plus'):
        super().__init__()
        if WavLMModel is None:
            raise ImportError('transformers is required for WavLM-Base+.')
        self.model = WavLMModel.from_pretrained(model_name)
        self.feature_dim = self.model.config.hidden_size

    def forward(self, waveform: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        if waveform.dim() == 3 and waveform.size(1) == 1:
            waveform = waveform.squeeze(1)
        elif waveform.dim() == 2 and waveform.size(0) == 1 and attention_mask is not None and attention_mask.dim() == 3:
            attention_mask = attention_mask.squeeze(1)
        outputs = self.model(input_values=waveform, attention_mask=attention_mask)
        return outputs.last_hidden_state.mean(dim=1)


print('WavLM-Base+ backbone ready')

## 8. Implement Cross-Attention Fusion Module

Cross-attention fuses video and audio embeddings before classification.

In [ ]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, video_dim: int, audio_dim: int, fusion_dim: int = 512, num_heads: int = 8):
        super().__init__()
        self.video_proj = nn.Linear(video_dim, fusion_dim)
        self.audio_proj = nn.Linear(audio_dim, fusion_dim)
        self.cross_attn_video = nn.MultiheadAttention(fusion_dim, num_heads, batch_first=True)
        self.cross_attn_audio = nn.MultiheadAttention(fusion_dim, num_heads, batch_first=True)
        self.norm_video = nn.LayerNorm(fusion_dim)
        self.norm_audio = nn.LayerNorm(fusion_dim)
        self.out_proj = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.GELU(),
            nn.Dropout(0.2),
        )

    def forward(self, video_features: torch.Tensor, audio_features: torch.Tensor) -> torch.Tensor:
        video_token = self.video_proj(video_features).unsqueeze(1)
        audio_token = self.audio_proj(audio_features).unsqueeze(1)
        video_ctx, _ = self.cross_attn_video(video_token, audio_token, audio_token)
        audio_ctx, _ = self.cross_attn_audio(audio_token, video_token, video_token)
        fused = torch.cat([self.norm_video(video_ctx.squeeze(1)), self.norm_audio(audio_ctx.squeeze(1))], dim=-1)
        return self.out_proj(fused)


print('Cross-attention fusion ready')

## 9. Construct Full Multimodal Model

This combines the two backbones, fusion block, and final classifier.

In [ ]:
class MultimodalDeepfakeModel(nn.Module):
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.video_backbone = VideoSwinTinyBackbone(pretrained=True)
        self.audio_backbone = WavLMBasePlusBackbone()
        self.fusion = CrossAttentionFusion(self.video_backbone.feature_dim, self.audio_backbone.feature_dim)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, video_frames: torch.Tensor, audio_waveform: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        video_features = self.video_backbone(video_frames)
        audio_features = self.audio_backbone(audio_waveform, attention_mask=attention_mask)
        fused_features = self.fusion(video_features, audio_features)
        return self.classifier(fused_features)


model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(DEVICE)
print(model.__class__.__name__)

## 10. Implement Data Augmentation Strategies

This notebook keeps augmentation hooks lightweight and easy to extend.

In [ ]:
def apply_video_augmentations(frames: torch.Tensor) -> torch.Tensor:
    if frames.size(0) > 1 and random.random() < 0.5:
        frames = torch.roll(frames, shifts=random.randint(0, frames.size(0) - 1), dims=0)
    return frames


def apply_audio_augmentations(waveform: torch.Tensor) -> torch.Tensor:
    if random.random() < 0.5:
        waveform = waveform + 0.001 * torch.randn_like(waveform)
    return waveform


print('Augmentation hooks ready')

## 11. Create Stratified DataLoader with Class-Balanced Sampling

The split is speaker-aware and the sampler balances the weaker classes.

In [ ]:
CLASS_TO_INDEX = {name: idx for idx, name in enumerate(CONFIG['class_names'])}


def build_class_weight_tensor(records: Sequence[Dict[str, str]], class_names: Sequence[str]) -> torch.Tensor:
    """Create inverse-frequency class weights for CrossEntropyLoss."""
    label_counts = Counter(record['label'] for record in records)
    total = max(1, len(records))
    num_classes = max(1, len(class_names))
    weights: List[float] = []
    for class_name in class_names:
        class_count = label_counts.get(class_name, 0)
        if class_count == 0:
            weights.append(0.0)
        else:
            weights.append(total / (num_classes * class_count))
    return torch.tensor(weights, dtype=torch.float32)


class DeepfakeDataset(Dataset):
    def __init__(self, records: Sequence[Dict[str, str]], train_mode: bool = True):
        self.records = list(records)
        self.train_mode = train_mode

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        record = self.records[idx]
        video_frames = preprocess_video_clip(record['path'], train_mode=self.train_mode)
        audio_waveform = preprocess_audio_clip(record['path'], train_mode=self.train_mode)
        label = CLASS_TO_INDEX[record['label']]
        return {'video_frames': video_frames, 'audio_waveform': audio_waveform, 'label': label, 'speaker_id': record['speaker_id'], 'path': record['path']}


def build_speaker_stratified_splits(records: Sequence[Dict[str, str]]) -> Tuple[List[Dict[str, str]], List[Dict[str, str]], List[Dict[str, str]]]:
    speaker_to_records: Dict[str, List[Dict[str, str]]] = defaultdict(list)
    for record in records:
        speaker_to_records[record['speaker_id']].append(record)
    speaker_rows = []
    for speaker_id, speaker_records in speaker_to_records.items():
        dominant_label = Counter(item['label'] for item in speaker_records).most_common(1)[0][0]
        speaker_rows.append({'speaker_id': speaker_id, 'label': dominant_label})
    frame = pd.DataFrame(speaker_rows)
    if frame.empty:
        return [], [], []

    n_speakers = len(frame)
    n_classes = len(CONFIG['class_names'])
    label_counts = frame['label'].value_counts()
    can_stratify = n_speakers >= 2 * n_classes and int(label_counts.min()) >= 2

    train_speakers = val_speakers = test_speakers = None

    if can_stratify:
        first_test_size = min(max(n_classes, n_speakers // 4), n_speakers - n_classes)
        if first_test_size >= n_classes and (n_speakers - first_test_size) >= n_classes:
            train_speakers, temp_speakers = train_test_split(
                frame,
                test_size=first_test_size,
                random_state=CONFIG['seed'],
                stratify=frame['label'],
            )
            temp_size = len(temp_speakers)
            if temp_size >= 2 * n_classes:
                val_speakers, test_speakers = train_test_split(
                    temp_speakers,
                    test_size=temp_size // 2,
                    random_state=CONFIG['seed'],
                    stratify=temp_speakers['label'],
                )
            else:
                temp_speakers = temp_speakers.sample(frac=1, random_state=CONFIG['seed']).reset_index(drop=True)
                mid = max(1, temp_size // 2)
                val_speakers = temp_speakers.iloc[:mid]
                test_speakers = temp_speakers.iloc[mid:]
        else:
            can_stratify = False

    if not can_stratify:
        shuffled = frame.sample(frac=1, random_state=CONFIG['seed']).reset_index(drop=True)
        n_train = max(1, int(round(n_speakers * 0.7)))
        n_val = max(1, int(round(n_speakers * 0.15)))
        if n_train + n_val >= n_speakers:
            n_val = max(1, n_speakers - n_train - 1)
        train_speakers = shuffled.iloc[:n_train]
        val_speakers = shuffled.iloc[n_train:n_train + n_val]
        test_speakers = shuffled.iloc[n_train + n_val:]

    train_ids = set(train_speakers['speaker_id'])
    val_ids = set(val_speakers['speaker_id'])
    test_ids = set(test_speakers['speaker_id'])
    train_records = [record for record in records if record['speaker_id'] in train_ids]
    val_records = [record for record in records if record['speaker_id'] in val_ids]
    test_records = [record for record in records if record['speaker_id'] in test_ids]
    return train_records, val_records, test_records


def build_class_balanced_sampler(records: Sequence[Dict[str, str]]) -> WeightedRandomSampler:
    if not records:
        raise ValueError('Cannot build a class-balanced sampler from an empty record list.')
    label_counts = Counter(record['label'] for record in records)
    weights = [1.0 / label_counts[record['label']] for record in records]
    return WeightedRandomSampler(weights=weights, num_samples=len(records), replacement=True)


def build_dataloaders(records: Sequence[Dict[str, str]]) -> Tuple[DataLoader, DataLoader, DataLoader]:
    train_records, val_records, test_records = build_speaker_stratified_splits(records)
    train_dataset = DeepfakeDataset(train_records, train_mode=True)
    val_dataset = DeepfakeDataset(val_records, train_mode=False)
    test_dataset = DeepfakeDataset(test_records, train_mode=False)
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], sampler=build_class_balanced_sampler(train_records), num_workers=CONFIG['num_workers'])
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
    return train_loader, val_loader, test_loader


print('Dataloaders ready')

## 12. Implement Training Loop with Mixed Precision

The training loop uses AMP and gradient accumulation for memory efficiency.

In [ ]:
def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {
        'video_frames': batch['video_frames'].to(device),
        'audio_waveform': batch['audio_waveform'].to(device),
        'label': batch['label'].to(device),
    }


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, scaler: GradScaler, criterion: nn.Module, device: torch.device, accumulation_steps: int = 1, stage_name: str = 'train') -> float:
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    total_batches = max(1, len(loader))
    autocast_device = 'cuda' if device.type == 'cuda' else 'cpu'
    for step, batch in enumerate(loader, start=1):
        batch = move_batch_to_device(batch, device)
        with autocast(autocast_device, enabled=CONFIG['use_amp']):
            logits = model(batch['video_frames'], batch['audio_waveform'])
            loss = criterion(logits, batch['label']) / accumulation_steps
        scaler.scale(loss).backward()
        if step % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        running_loss += loss.item() * accumulation_steps
        if step == 1 or step % max(1, total_batches // 5) == 0 or step == total_batches:
            print(f'[{stage_name}] batch {step}/{total_batches} loss={loss.item() * accumulation_steps:.4f}')
    if total_batches % accumulation_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
    return running_loss / total_batches


print('Training loop ready')

## 13. Stage A: Train Unimodal Branches (Frozen Backbones)

This stage freezes the backbones and trains only the light adaptation layers first.

In [ ]:
def freeze_backbones(model: MultimodalDeepfakeModel) -> None:
    for parameter in model.video_backbone.parameters():
        parameter.requires_grad = False
    for parameter in model.audio_backbone.parameters():
        parameter.requires_grad = False


def unfreeze_top_blocks(model: MultimodalDeepfakeModel) -> None:
    freeze_backbones(model)
    for parameter in model.fusion.parameters():
        parameter.requires_grad = True
    for parameter in model.classifier.parameters():
        parameter.requires_grad = True


def evaluate_model(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> Dict[str, float]:
    model.eval()
    y_true, y_pred, y_prob, losses = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch, device)
            logits = model(batch['video_frames'], batch['audio_waveform'])
            loss = criterion(logits, batch['label'])
            losses.append(loss.item())
            probs = torch.softmax(logits, dim=1)
            y_prob.extend(probs.detach().cpu().tolist())
            y_pred.extend(logits.argmax(dim=1).detach().cpu().tolist())
            y_true.extend(batch['label'].detach().cpu().tolist())
    roc_auc = 0.0
    if y_true and len(set(y_true)) > 1:
        try:
            roc_auc = roc_auc_score(y_true, np.array(y_prob), multi_class='ovr', average='macro')
        except Exception:
            roc_auc = 0.0
    return {
        'loss': float(np.mean(losses)) if losses else 0.0,
        'accuracy': accuracy_score(y_true, y_pred) if y_true else 0.0,
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0) if y_true else 0.0,
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0) if y_true else 0.0,
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0) if y_true else 0.0,
        'roc_auc_macro_ovr': roc_auc,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist() if y_true else [],
        'classification_report': classification_report(y_true, y_pred, target_names=CONFIG['class_names'], zero_division=0) if y_true else '',
    }


def bundle_outputs(files: Sequence[str], bundle_name: str) -> Path:
    import shutil
    tmpdir = WORK_DIR / (bundle_name + '_bundle')
    if tmpdir.exists():
        shutil.rmtree(tmpdir)
    tmpdir.mkdir(parents=True, exist_ok=True)
    for p in files:
        try:
            src = Path(p)
            if src.exists():
                shutil.copy2(str(src), str(tmpdir / src.name))
        except Exception as e:
            print('Warning copying', p, '->', e)
    archive_base = str(WORK_DIR / bundle_name)
    shutil.make_archive(archive_base, 'zip', root_dir=str(tmpdir))
    shutil.rmtree(tmpdir)
    archive_path = Path(archive_base + '.zip')
    print('Created bundle:', archive_path)
    return archive_path


def save_stage_checkpoint(stage_name: str, epoch: int, model: nn.Module, optimizer: torch.optim.Optimizer, scaler: GradScaler, best_metrics: Dict[str, float]) -> Path:
    checkpoint_path = WORK_DIR / f'{stage_name}_resume.pt'
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_metrics': best_metrics,
        'config': CONFIG,
    }, checkpoint_path)
    return checkpoint_path


def load_stage_checkpoint(stage_name: str, model: nn.Module, optimizer: torch.optim.Optimizer, scaler: GradScaler, device: torch.device) -> Tuple[int, Dict[str, float]]:
    checkpoint_path = WORK_DIR / f'{stage_name}_resume.pt'
    if not CONFIG['resume_from_checkpoints'] or not checkpoint_path.exists():
        return 1, {'f1_macro': -1.0}

    checkpoint = torch.load(str(checkpoint_path), map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    best_metrics = checkpoint.get('best_metrics', {'f1_macro': -1.0})
    start_epoch = int(checkpoint.get('epoch', 0)) + 1
    print(f'[{stage_name}] resumed from epoch {start_epoch - 1} using {checkpoint_path}')
    return start_epoch, best_metrics


def run_stage(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, stage_name: str, epochs: int, lr: float, device: torch.device) -> Dict[str, float]:
    train_records = getattr(getattr(train_loader, 'dataset', None), 'records', [])
    class_weights = build_class_weight_tensor(train_records, CONFIG['class_names']).to(device) if train_records else None
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=CONFIG['weight_decay'])
    scaler = GradScaler(device='cuda' if device.type == 'cuda' else 'cpu', enabled=CONFIG['use_amp'])
    start_epoch, best_metrics = load_stage_checkpoint(stage_name, model, optimizer, scaler, device)
    print(f'--- {stage_name} start ---')
    if class_weights is not None:
        print(f'[{stage_name}] class weights:', class_weights.detach().cpu().tolist())
    for epoch in range(start_epoch, epochs + 1):
        print(f'[{stage_name}] epoch {epoch}/{epochs} started')
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, criterion, device, CONFIG['accumulation_steps'], stage_name=stage_name)
        val_metrics = evaluate_model(model, val_loader, criterion, device)
        print(f"[{stage_name}] epoch={epoch} train_loss={train_loss:.4f} val_f1={val_metrics['f1_macro']:.4f} val_acc={val_metrics['accuracy']:.4f}")
        if CONFIG['checkpoint_every_epoch']:
            save_stage_checkpoint(stage_name, epoch, model, optimizer, scaler, best_metrics)
        if val_metrics['f1_macro'] > best_metrics['f1_macro']:
            best_metrics = val_metrics
            save_stage_checkpoint(stage_name, epoch, model, optimizer, scaler, best_metrics)
            # save best checkpoint
            best_ckpt = WORK_DIR / f"{stage_name}_best.pth"
            try:
                torch.save(model.state_dict(), str(best_ckpt))
                print(f'[{stage_name}] saved best checkpoint: {best_ckpt}')
            except Exception as e:
                print(f'[{stage_name}] failed to save best checkpoint:', e)
            # save interim metrics and optionally bundle partial outputs
            try:
                interim_metrics = WORK_DIR / f"{stage_name}_best_metrics.json"
                with interim_metrics.open('w', encoding='utf-8') as f:
                    json.dump(val_metrics, f, indent=2)
                if CONFIG['full_dataset_enabled']:
                    bundle_files = [str(WORK_DIR / 'full_run_manifest.json'), str(interim_metrics), str(best_ckpt)]
                else:
                    bundle_files = [str(WORK_DIR / 'pilot_manifest_970.json'), str(interim_metrics), str(best_ckpt)]
                bundle_outputs(bundle_files, f'{stage_name}_partial')
            except Exception as e:
                print(f'[{stage_name}] partial bundle failed:', e)
    # end epochs
    final_ckpt = WORK_DIR / f"{stage_name}_final.pth"
    try:
        torch.save(model.state_dict(), str(final_ckpt))
        print(f'[{stage_name}] saved final checkpoint: {final_ckpt}')
    except Exception as e:
        print(f'[{stage_name}] failed to save final checkpoint:', e)
    # save best metrics summary
    metrics_path = WORK_DIR / f"{stage_name}_metrics.json"
    with metrics_path.open('w', encoding='utf-8') as f:
        json.dump(best_metrics, f, indent=2)
    # final bundle for the stage
    try:
        if CONFIG['full_dataset_enabled']:
            bundle_files = [str(WORK_DIR / 'full_run_manifest.json'), str(metrics_path), str(final_ckpt)]
        else:
            bundle_files = [str(WORK_DIR / 'pilot_manifest_970.json'), str(metrics_path), str(final_ckpt)]
        bundle_outputs(bundle_files, f'{stage_name}_outputs')
    except Exception as e:
        print(f'[{stage_name}] final bundle failed:', e)
    print(f'--- {stage_name} complete ---')
    return best_metrics


def stage_a_train(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    freeze_backbones(model)
    return run_stage(model, train_loader, val_loader, 'Stage A', CONFIG['epochs_stage_a'], CONFIG['lr_stage_a'], device)


print('Stage A ready')

## 14. Stage B: Fine-tune Branch Top Blocks

This stage unfreezes the final parts of the model and lowers the learning rate.

In [ ]:
def stage_b_finetune(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    unfreeze_top_blocks(model)
    return run_stage(model, train_loader, val_loader, 'Stage B', CONFIG['epochs_stage_b'], CONFIG['lr_stage_b'], device)


print('Stage B ready')

## 15. Stage C: Train Fusion with Partial Freezing

The fusion stage emphasizes cross-attention while keeping the backbones mostly fixed.

In [ ]:
def stage_c_train_fusion(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    freeze_backbones(model)
    return run_stage(model, train_loader, val_loader, 'Stage C', CONFIG['epochs_stage_c'], CONFIG['lr_stage_c'], device)


print('Stage C ready')

## 16. Stage D: End-to-End Low Learning-Rate Fine-tuning

This is the final polishing step after the staged pretraining and fusion steps.

In [ ]:
def stage_d_end_to_end(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    unfreeze_top_blocks(model)
    return run_stage(model, train_loader, val_loader, 'Stage D', CONFIG['epochs_stage_d'], CONFIG['lr_stage_d'], device)


print('Stage D ready')

## 17. Evaluation and Metrics

The evaluation block computes the core classification metrics and confusion matrix.

In [ ]:
def evaluate_and_save(model: MultimodalDeepfakeModel, test_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    criterion = nn.CrossEntropyLoss()
    metrics = evaluate_model(model, test_loader, criterion, device)
    metrics_path = WORK_DIR / 'pilot_metrics.json'
    with metrics_path.open('w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2)
    print('Saved metrics to', metrics_path)
    return metrics


print('Evaluation ready')

## 18. Scope for Full Dataset Training

When the pilot is stable, switch `FULL_DATASET_MODE = True` and keep the same staged training plan for the entire corpus.

Scale-up notes:
- keep speaker-aware splitting
- keep class-balanced sampling
- keep mixed precision and gradient accumulation
- save checkpoints after each stage
- use low learning rates for the final fine-tune

In [ ]:
def run_pilot_training_pipeline() -> Optional[Dict[str, float]]:
    required_names = [
        'stage_a_train',
        'stage_b_finetune',
        'stage_c_train_fusion',
        'stage_d_end_to_end',
        'evaluate_and_save',
        'build_dataloaders',
        'MultimodalDeepfakeModel',
    ]
    missing_names = [name for name in required_names if name not in globals()]
    if missing_names:
        print('Training setup is not ready yet.')
        print('Run the earlier notebook cells first, then rerun this cell.')
        print('Missing names:', ', '.join(missing_names))
        return None

    run_training = False
    pilot_train_loader, pilot_val_loader, pilot_test_loader = build_dataloaders(pilot_records)
    pilot_model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(DEVICE)
    print('Pilot loaders built:', len(pilot_train_loader), len(pilot_val_loader), len(pilot_test_loader))

    if run_training:
        stage_a_metrics = stage_a_train(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        stage_b_metrics = stage_b_finetune(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        stage_c_metrics = stage_c_train_fusion(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        stage_d_metrics = stage_d_end_to_end(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        final_metrics = evaluate_and_save(pilot_model, pilot_test_loader, DEVICE)
        print(final_metrics)
        return final_metrics

    print('Set RUN_TRAINING = True inside this cell to execute the pilot training pipeline.')

    def bundle_downloadable_outputs(work_dir: Path) -> List[Path]:
        bundles: List[Path] = []
        output_groups = {
            'pilot_manifest': [work_dir / 'pilot_manifest_970.json'],
            'pilot_metrics': [work_dir / 'pilot_metrics.json'],
        }
        for bundle_name, files in output_groups.items():
            bundle_path = work_dir / f'{bundle_name}.zip'
            with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
                for file_path in files:
                    if file_path.exists():
                        archive.write(file_path, arcname=file_path.name)
            bundles.append(bundle_path)
        return bundles

    def display_download_links(paths: Sequence[Path]) -> None:
        for path in paths:
            if path.exists():
                print('Download:', path)
                try:
                    display(FileLink(str(path)))
                except Exception:
                    pass

    bundle_paths = bundle_downloadable_outputs(WORK_DIR)
    display_download_links(bundle_paths)
    print('Downloadable artifacts prepared:', [path.name for path in bundle_paths])
    return None


run_pilot_training_pipeline()

## Downloadable Outputs

The notebook writes pilot artifacts into the Kaggle working directory and bundles them as zip files for download at the end of the run. If you execute the training stages, the final cell will generate clickable links for the manifest and metrics bundles.

## 19. Test a Single Video After Cross-Domain Fusion

Set the video path and checkpoint path below to run fused video+audio inference on one clip.

In [ ]:
# Provide your test video path here (Kaggle example: /kaggle/input/<dataset>/<subdir>/clip.mp4)
TEST_VIDEO_PATH = ''

# Use Stage D checkpoint if available, otherwise Stage C, then fallback to current in-memory model
CHECKPOINT_CANDIDATES = [
    WORK_DIR / 'Stage D_final.pth',
    WORK_DIR / 'Stage D_best.pth',
    WORK_DIR / 'Stage C_final.pth',
    WORK_DIR / 'Stage C_best.pth',
]


def get_best_checkpoint_path(candidates: Sequence[Path]) -> Optional[Path]:
    for path in candidates:
        if path.exists():
            return path
    return None


def load_model_for_inference(device: torch.device, checkpoint_path: Optional[Path] = None) -> MultimodalDeepfakeModel:
    inference_model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(device)
    if checkpoint_path is not None and checkpoint_path.exists():
        state = torch.load(str(checkpoint_path), map_location=device)
        inference_model.load_state_dict(state, strict=True)
        print('Loaded checkpoint:', checkpoint_path)
    else:
        print('No checkpoint found. Using current model weights.')
    inference_model.eval()
    return inference_model


@torch.no_grad()
def predict_single_video(video_path: str, model: MultimodalDeepfakeModel, device: torch.device) -> Dict[str, object]:
    video_frames = preprocess_video_clip(video_path, train_mode=False).unsqueeze(0).to(device)
    audio_waveform = preprocess_audio_clip(video_path, train_mode=False).to(device)
    logits = model(video_frames, audio_waveform)
    probs = torch.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()
    pred_idx = int(np.argmax(probs))
    pred_label = CONFIG['class_names'][pred_idx]
    prob_map = {CONFIG['class_names'][i]: float(probs[i]) for i in range(len(CONFIG['class_names']))}
    return {
        'predicted_class': pred_label,
        'confidence': float(probs[pred_idx]),
        'probabilities': prob_map,
    }


if TEST_VIDEO_PATH.strip():
    test_path = Path(TEST_VIDEO_PATH)
    if not test_path.exists():
        print('Video not found:', test_path)
    else:
        best_ckpt = get_best_checkpoint_path(CHECKPOINT_CANDIDATES)
        infer_model = load_model_for_inference(DEVICE, best_ckpt)
        result = predict_single_video(str(test_path), infer_model, DEVICE)
        print('Prediction:', result['predicted_class'])
        print('Confidence:', round(result['confidence'], 4))
        print('Class probabilities:', json.dumps(result['probabilities'], indent=2))
else:
    print('Set TEST_VIDEO_PATH to run single-video inference after fusion training.')